# Развёртка макросов и AST

Команды компилируют отдельный фрагмент, но не выполняют его runtime-код. Compile-time макросы запускаются как обычно. Представление получено после вывода типов, до оптимизации.

In [1]:
%%writefile notebook_macros.das
options gen2
module notebook_macros public
require daslib/ast
require daslib/ast_boost
require daslib/templates_boost
require daslib/macro_boost
require daslib/ast_verify
[call_macro(name="plus_one")]
class PlusOneMacro : AstCallMacro {
    def override visit(prog:ProgramPtr; mod:Module?; var expr:ExprCallMacro?):ExpressionPtr {
        macro_verify(length(expr.arguments)==1, prog, expr.at, "plus_one expects one argument")
        var result <- qmacro($e(clone_expression(expr.arguments[0])) + 1)
        verify_expression(result)
        return <- result
    }
}


Wrote C:\src\daslang-jupyter\notebook_macros.das


In [2]:
require ./notebook_macros.das
var value=40
var calls=0
def bump():int {calls+=1;return calls}
def sample(x:int):int {return plus_one(x)}

In [4]:
%expand plus_one(value)

// After type inference and macro expansion; optimizer disabled.

/*FULLY INFERRED*/
// [access_global]
[export]
[$::export]
def public inspected_cell : int const {
	return value + 1;
}



In [5]:
%%ast
plus_one(value)

// After type inference and macro expansion; optimizer disabled.
Function <cell>
  ExprBlock
    ExprReturn : void
      ExprOp2 : int const
        ExprRef2Value : int -const
          ExprVar : int& -const  value
        ExprConstInt : int const  1

In [6]:
%expand sample

// After type inference and macro expansion; optimizer disabled.

/*FULLY INFERRED*/
[nosideeffects]
[export]
[$::export]
def public sample(x:int const) : int const {
	return x + 1;
}



In [7]:
%%expand
def temporary(x:int):int {return plus_one(x)}

// After type inference and macro expansion; optimizer disabled.

/*FULLY INFERRED*/
[nosideeffects]
[export]
[$::export]
def public inspected_cell : void {
}



/*FULLY INFERRED*/
[nosideeffects]
def public temporary(x:int const) : int const {
	return x + 1;
}



In [8]:
%%ast
value=999
var temporary=plus_one(value)
print("THIS MUST NOT RUN")

// After type inference and macro expansion; optimizer disabled.
Function <cell>
  ExprBlock
    ExprCopy : void
      ExprVar : int& -const  value
      ExprConstInt : int const  999
    ExprLet
      ExprOp2 : int const
        ExprRef2Value : int -const
          ExprVar : int& -const  value
        ExprConstInt : int const  1
    ExprCall : void
      ExprConstString : string const  "THIS MUST NOT RUN"
      ExprFakeContext : __context const  __context__
      ExprFakeLineInfo : __lineInfo const  __lineinfo__
    ExprReturn : void

In [9]:
%expand bump()

// After type inference and macro expansion; optimizer disabled.

/*FULLY INFERRED*/
// [access_global]
[export]
[$::export]
def public inspected_cell : int const {
	return bump();
}



Переменная `value` должна остаться 40, `calls` — 0. Функция `temporary` и локальные переменные из просмотренных фрагментов в сессию не добавляются.

In [10]:
value

40

In [11]:
calls

0

In [12]:
%%ast
struct Point {x:int;y:float}

// After type inference and macro expansion; optimizer disabled.
Function <cell>
  ExprBlock
Structure Point
  Field x : int
  Field y : float

In [13]:
%reset